# Prompt Chaining Workflow: Blog Generation

## Overview

This workflow uses **Prompt Chaining** to generate a high-quality blog post in multiple stages. Each step builds upon the output of the previous step, allowing for better structure, consistency, and quality control.

### Workflow Steps

1. **Generate Blog Outline**
2. **Generate Blog Content**
3. **Evaluate Blog Against Outline**

---

## Architecture

```text
User Topic
    │
    ▼
┌─────────────────────┐
│  Prompt 1           │
│ Generate Outline    │
└─────────┬───────────┘
          │
          ▼
      Blog Outline
          │
          ▼
┌─────────────────────┐
│  Prompt 2           │
│ Generate Blog       │
│ Using Outline       │
└─────────┬───────────┘
          │
          ▼
      Blog Draft
          │
          ▼
┌─────────────────────┐
│  Prompt 3           │
│ Rate Blog Against   │
│ Outline             │
└─────────┬───────────┘
          │
          ▼
    Evaluation Report
```

# Benefits of Prompt Chaining

- Produces more structured content.
- Improves consistency across long-form writing.
- Enables validation and quality control.
- Reduces hallucinations by grounding content in an approved outline.
- Makes debugging and optimization easier by isolating each step.


In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [25]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rate: int

In [11]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f"Generate an outline for a blog on the topic - {title}"

    outline_content = model.invoke(prompt).content

    state['outline'] = outline_content

    return state

In [20]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed blog on the topic - {title} using the following outline \n {outline}"

    blog_content = model.invoke(prompt).content

    state['content'] = blog_content

    return state

In [26]:
def rate_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f"Provide me just rating (integer ONLY) Based on the created outline \n {outline} for the give title - {title} rate the blog generted \n {content} out of 10"

    rate = model.invoke(prompt).content

    state['rate'] = rate

    return state

In [27]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('rate_blog', rate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'rate_blog')
graph.add_edge('rate_blog', END)

workflow = graph.compile()

In [28]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here\'s a suggested outline for a blog on the "Rise of AI in India":\n\n**I. Introduction**\n\n* Brief overview of Artificial Intelligence (AI) and its growing importance globally\n* Context: India\'s emergence as a significant player in the global AI landscape\n* Thesis statement: India is witnessing a significant rise in AI adoption, driven by government initiatives, technological advancements, and innovative startups.\n\n**II. Current State of AI in India**\n\n* Overview of India\'s AI ecosystem, including key players, industries, and applications\n* Statistics on AI adoption, investment, and job creation in India\n* Discussion of India\'s strengths and weaknesses in AI, including talent pool, infrastructure, and regulatory framework\n\n**III. Government Initiatives and Policies**\n\n* Overview of government initiatives to promote AI in India, such as:\n\t+ National AI Strategy\n\t+ AI Task Force\n\t+ Digital India program\n* Discussion o

In [29]:
print(final_state['outline'])

Here's a suggested outline for a blog on the "Rise of AI in India":

**I. Introduction**

* Brief overview of Artificial Intelligence (AI) and its growing importance globally
* Context: India's emergence as a significant player in the global AI landscape
* Thesis statement: India is witnessing a significant rise in AI adoption, driven by government initiatives, technological advancements, and innovative startups.

**II. Current State of AI in India**

* Overview of India's AI ecosystem, including key players, industries, and applications
* Statistics on AI adoption, investment, and job creation in India
* Discussion of India's strengths and weaknesses in AI, including talent pool, infrastructure, and regulatory framework

**III. Government Initiatives and Policies**

* Overview of government initiatives to promote AI in India, such as:
	+ National AI Strategy
	+ AI Task Force
	+ Digital India program
* Discussion of key policies and regulations, including data protection and privacy la

In [30]:
print(final_state['content'])

**The Rise of AI in India: A New Era of Innovation and Growth**

**Introduction**

Artificial Intelligence (AI) has been transforming the world at an unprecedented pace, and India is no exception. With its growing importance globally, AI has become a crucial aspect of various industries, from healthcare to finance, and from retail to manufacturing. As a significant player in the global AI landscape, India is witnessing a significant rise in AI adoption, driven by government initiatives, technological advancements, and innovative startups. In this blog, we will explore the current state of AI in India, the key drivers of its growth, and the opportunities and challenges that lie ahead.

**Current State of AI in India**

India's AI ecosystem is rapidly evolving, with key players, industries, and applications emerging across the country. According to a report by NASSCOM, the Indian AI market is expected to reach $7.8 billion by 2025, growing at a CAGR of 30.8%. The report also estimates th

In [31]:
print(final_state['rate'])

8
